In [3]:
from collections import namedtuple
from time import sleep
import logging

from pulao.logging import init_logging

import pandas as pd

from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pulao.constant import *

from pulao.logging import logger
init_logging(level=logging.DEBUG)
open("./logs/pulao.log", "w").close() # 每次运行前清空日志
logger.debug("开始运行")

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(100*3)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    o = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=o,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(symbol=bar.symbol, exchange=bar.exchange.value, interval=bar.interval.value, datetime=datetime, turnover=money, open_price=o, close_price=close, high_price=high, low_price=low, volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
tred_manager = TrendManager(swing_manager=swing_manager)
# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

logger.debug("运行结束")
# sleep(3) # 等一会，让日志输出完
#
# region . Plotly - Cbar
#
df_cbar = cbar_manager.df_cbar.to_pandas()
df_cbar["datetime"] = pd.date_range("2025-01-01", periods=df_cbar.shape[0], freq="h")
df_cbar["open_price"] = df_cbar["high_price"]
df_cbar["close_price"] = df_cbar["low_price"]
df_cbar.insert(0, "index", range(len(df_cbar)))

# region 构建波段数据
df_swing = swing_manager.df_swing.to_pandas()
df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing.insert(0, "index", range(len(df_swing)))

for i, row in enumerate(df_swing.itertuples()):
    df = df_cbar[(df_cbar["id"] >= row.start_id) & (df_cbar["id"] <= row.end_id)]
    start_datetime = df.iloc[0]["datetime"]
    end_datetime = df.iloc[-1]["datetime"]
    df_swing.at[i, "start_datetime"] = start_datetime
    df_swing.at[i, "end_datetime"] = end_datetime

xs = []
ys = []
texts = []
text_positions  = []
for i, row in enumerate(df_swing.itertuples()):
    xs += [row.start_datetime, row.end_datetime, None]
    start_index = df_cbar[(df_cbar["id"] == row.start_id)]["index"].iloc[0]
    end_index = df_cbar[(df_cbar["id"] == row.end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_cbar[(df_cbar["id"] == row.start_id)]["low_price"].iloc[0]
        end_price = df_cbar[(df_cbar["id"] == row.end_id)]["high_price"].iloc[0]
        text_positions += ['bottom center', 'top center', "middle center"]
    else:
        start_price = df_cbar[(df_cbar["id"] == row.start_id)]["high_price"].iloc[0]
        end_price = df_cbar[(df_cbar["id"] == row.end_id)]["low_price"].iloc[0]
        text_positions += ['top center', 'bottom center', "middle center"]
    ys += [start_price, end_price, None]
    texts += [start_index, end_index, None]


# endregion

fig = make_subplots(
    rows=1, cols=1
)
fig.add_trace(go.Candlestick(
    x=df_cbar['datetime'],
    open=df_cbar['open_price'],
    high=df_cbar['high_price'],
    low=df_cbar['low_price'],
    close=df_cbar['close_price'],
    name='K线',
), row=1, col=1)

df_fractal_bottom = df_cbar[(df_cbar['fractal_type'] == FractalType.BOTTOM)]

fig.add_trace(go.Scatter(
    x=df_fractal_bottom['datetime'],
    y=df_fractal_bottom['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers+text',
    name='swing point low',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#1E90FF',
    ),
    text=df_fractal_bottom["index"],      # ← 添加序号
    textposition="bottom center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_fractal_top = df_cbar[(df_cbar['fractal_type'] == FractalType.TOP)]

fig.add_trace(go.Scatter(
    x=df_fractal_top['datetime'],
    y=df_fractal_top['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers+text',
    name='swing point high',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#FF4500',
    ),
    text=df_fractal_top["index"],      # ← 添加序号
    textposition="top center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=xs,
    y=ys,
    name='swing segments',
    mode='lines+text',    # lines+text 支持显示文字
    line=dict(width=2, color='orange'),
    text=texts,
    textposition=text_positions,  # 文字位置
))

title = 'Swing Chart - 波段标识'
fig.update_layout(title=title,
                  height=900,
                  hovermode='x unified',  # X轴悬停联动虚线
                  )

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)

fig.show()

# endregion



{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2025-11-28 00:25:14"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-11-28 00:25:14"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-11-28 00:25:14"}
{"cbar_count": 2, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-11-28 00:25:14"}
{"fractal": "Fractal(left=CBar(id=119956736734920704, start_id=119956736722337792, end_id=119956736730726400, high_price=942.2349853515625, low_price=935.9240112304688, created_at=datetime.datetime(2025, 11, 28, 0, 25, 14, 726000), fractal_type=0), middle=CBar(id=119956736743309312, start_id=119956736743309312, end_id=119956736743309312, high_price=939.052001953125, low_price=930.2979736328125, created_at=datetime.datetime(2025, 11, 28, 0, 25, 14, 728000), fractal_type=-1), right=CBar(id=119956736755892224, start_id=119956736755892224, end_id=11995

In [4]:
# 测试swing 构建算法
from IPython.display import display
from pulao.logging import logger,init_logging
import logging
from time import sleep
from pulao.swing import Swing,SwingManager
from pulao.bar import CBar,CBarManager,SBarManager
from pulao.constant import EventType
init_logging(level=logging.DEBUG)
with open("./logs/pulao.log", "w"):
    pass # 每次运行前清空日志

logger.debug("开始运行")

def format_df_swing():
    # 在df_swing中显示df_cbar的索引顺序
    import polars as pl
    if "index" in cbar_manager.df_cbar.columns:
        cbar_manager.df_cbar = cbar_manager.df_cbar.drop("index")
    cbar_manager.df_cbar = cbar_manager.df_cbar.with_row_index("index")

    df_cbar = cbar_manager.df_cbar.select(["id","index"])
    df1 = swing_manager.df_swing.join(df_cbar, left_on="start_id", right_on="id", how="left")
    df2 = swing_manager.df_swing.join(df_cbar, left_on="end_id", right_on="id", how="left")
    df1 = df1.rename({"index":"start_index"})
    df2 = df2.rename({"index":"end_index"})
    df2 = df2.select(["id","end_index"])
    df1 = df1.join(df2, on="id", how="left")
    df = df1.select(["id","start_index","end_index","start_id","end_id","high_price","low_price","direction","is_completed","created_at"])

    return df


cbar_manager = CBarManager(sbar_manager=SBarManager())
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)
df_cbar = cbar_manager.read_parquet()
cbar_manager.df_cbar = cbar_manager.df_cbar.slice(0,0)
for i in range(df_cbar.height-1):
    cbar = CBar(**df_cbar.row(i, named=True))
    cbar_manager._append_cbar(start_id=cbar.start_id, end_id=cbar.end_id, high_price=cbar.high_price, low_price=cbar.low_price,fractal_type=cbar.fractal_type)
    cbar_manager.notify(EventType.CBAR_CREATED)

logger.debug("运行结束")
sleep(0.2)


display("df_swing")
df = format_df_swing()
display(df)

display("df_cbar")
cbar_manager.df_cbar



{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2025-11-28 00:28:53"}
{"event": "运行结束", "level": "debug", "logger": "__main__", "time": "2025-11-28 00:28:53"}


'df_swing'

id,start_index,end_index,start_id,end_id,high_price,low_price,direction,is_completed,created_at
u64,u32,u32,u64,u64,f32,f32,i8,bool,datetime[ms]
119956737053687808,1,24,119956736743309312,119956737041104896,988.528015,930.297974,1,true,2025-11-28 00:25:14.802
119956738098069504,24,98,119956737041104896,119956738068709376,988.528015,842.133972,-1,true,2025-11-28 00:25:15.051
119956738186149888,98,104,119956738068709376,119956738169372672,879.856995,842.133972,1,true,2025-11-28 00:25:15.072
119956738697854976,104,135,119956738169372672,119956738672689152,879.856995,834.388,-1,true,2025-11-28 00:25:15.194
119956738886598656,135,144,119956738672689152,119956738853044224,873.119019,841.734985,1,true,2025-11-28 00:25:15.239
…,…,…,…,…,…,…,…,…,…
119956749716291584,635,641,119956749519159296,119956749686931456,782.034973,752.098022,1,true,2025-11-28 00:25:17.821
119956750051835904,641,653,119956749686931456,119956750022475776,782.034973,739.611023,-1,true,2025-11-28 00:25:17.901
119956750534180864,653,672,119956750022475776,119956750504820736,793.218018,739.611023,1,true,2025-11-28 00:25:18.016


'df_cbar'

index,id,start_id,end_id,high_price,low_price,fractal_type,created_at
u32,u64,u64,u64,f32,f32,i8,datetime[ms]
0,119956736734920704,119956736722337792,119956736730726400,942.234985,935.924011,0,2025-11-28 00:25:14.726
1,119956736743309312,119956736743309312,119956736743309312,939.052002,930.297974,-1,2025-11-28 00:25:14.728
2,119956736755892224,119956736755892224,119956736755892224,951.27301,937.492981,0,2025-11-28 00:25:14.731
3,119956736785252352,119956736768475136,119956736781058048,955.56897,948.888,1,2025-11-28 00:25:14.738
4,119956736793640960,119956736793640960,119956736793640960,950.607971,946.338989,-1,2025-11-28 00:25:14.740
…,…,…,…,…,…,…,…
755,119956752702636032,119956752673275904,119956752690053120,741.554016,735.406982,0,2025-11-28 00:25:18.533
756,119956752719413248,119956752711024640,119956752711024640,747.684021,740.830994,0,2025-11-28 00:25:18.537
757,119956752736190464,119956752731996160,119956752731996160,751.919006,744.698975,0,2025-11-28 00:25:18.541


In [ ]:
# 可视化日志
import pandas as pd
df = pd.read_json("./logs/pulao.log",lines=True)
df = df.drop(columns=["logger","level","time"])
priority_cols = ["event"]
other_cols = [c for c in df.columns if c not in priority_cols]
df = df[priority_cols+other_cols]
df["info"] = df[other_cols].apply(lambda row: row.dropna().tolist(), axis=1)
df = df.drop(columns=other_cols)
df["info"] = df["info"].apply(lambda x: "" if x == [] else x)
df

In [ ]:
cbar_manager.read_parquet()